# Planning Lab

In [1]:
!pip install unified-planning==1.1.0
!pip install up_fast_downward==0.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.6/706.6 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 21.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [unified-planning][unified-planning]

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.5 MB/s  0:00:00

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install matplotlib==3.7.1  # for visualisation in this notebook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 13.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 8.1 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]

[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from unified_planning.shortcuts import *

import unified_planning as up

up.shortcuts.get_environment().credits_stream = None

## Part (a): Model the Task

The code below models a simpler version of the Household Robot domain with two rooms and one open door between them. The robot is in room A initially and needs to move to room B. You can use it as a starting point for your own solution to the full Household Robot task.

In [ ]:

def get_planning_task():
    # Declare user types.
    Room = UserType("Room")
    Door = UserType("Door")
    Key  = UserType("Key")

    # Declare predicates.
    robot_in = up.model.Fluent("robot_in", BoolType(), r=Room)
    connected = up.model.Fluent("connected", BoolType(), r1=Room, d=Door, r2=Room)
    room_has_door = up.model.Fluent("room_has_door", BoolType(), r=Room, d=Door)

    key_in = up.model.Fluent("key_in", BoolType(), k=Key, r=Room)
    holding = up.model.Fluent("holding", BoolType(), k=Key)
    door_unlocked = up.model.Fluent("door_unlocked", BoolType(), d=Door)
    fits = up.model.Fluent("fits", BoolType(), k=Key, d=Door)


    # Add (typed) objects to problem.
    problem = up.model.Problem("household")

    living_room = up.model.Object("living_room", Room)
    corridor = up.model.Object("corridor", Room)
    kitchen = up.model.Object("kitchen", Room)
    bathroom = up.model.Object("bathroom", Room)
    lobby = up.model.Object("lobby", Room)

    door_k = up.model.Object("door_k", Door)
    door_l = up.model.Object("door_l", Door)
    door_b = up.model.Object("door_b", Door)
    door_c = up.model.Object("door_c", Door)
    door_f = up.model.Object("door_f", Door)

    key_all = up.model.Object("key_all", Key)
    key_l = up.model.Object("key_l", Key)
    key_k = up.model.Object("key_k", Key)
    key_b = up.model.Object("key_b", Key)
    key_c = up.model.Object("key_c", Key)
    key_f = up.model.Object("key_f", Key)
    
    rooms = [living_room, corridor, kitchen, bathroom, lobby]
    doors = [door_k, door_l, door_b, door_c, door_f]    
    keys = [key_all, key_l, key_k, key_b, key_c,  key_f]

    problem.add_objects(rooms)
    problem.add_objects(doors)
    problem.add_objects(keys)
    
    connections = [
        (living_room, door_k, kitchen),
        (living_room, door_l, corridor),
        (corridor, door_b, bathroom),
        (corridor, door_c, lobby)
    ]

    rooms_and_doors = []
    for room1, door, room2 in connections:
        rooms_and_doors.append((room1, door))
        rooms_and_doors.append((room2, door))


    # Specify the initial state.

    for f in [robot_in, connected, room_has_door, key_in, holding, door_unlocked, fits]:
        problem.add_fluent(f, default_initial_value=False)

    problem.set_initial_value(robot_in(living_room), True)

    problem.set_initial_value(key_in(key_k, living_room), True)
    problem.set_initial_value(key_in(key_l, living_room), True)
    problem.set_initial_value(key_in(key_all, kitchen), True)
    problem.set_initial_value(key_in(key_b, corridor), True)
    problem.set_initial_value(key_in(key_f, bathroom), True)
    problem.set_initial_value(key_in(key_c, bathroom), True)

    for room1, door, room2 in connections:
        problem.set_initial_value(connected(room1, door, room2), True)
        problem.set_initial_value(connected(room2, door, room1), True)

    for room, door in rooms_and_doors:
        problem.set_initial_value(room_has_door(room, door), True)

    problem.set_initial_value(door_unlocked(door_f), True)

    ### TODO Lägg till fits init
  

    # Add actions.
    move = up.model.InstantaneousAction("move", room1=Room, door=Door, room2=Room)
    room1 = move.parameter("room1")
    door = move.parameter("door")
    room2 = move.parameter("room2")
    move.add_precondition(robot_in(room1))
    move.add_precondition(connected(room1, door, room2))
    move.add_effect(robot_in(room1), False)
    move.add_effect(robot_in(room2), True)
    problem.add_action(move)

    # Specify the goal.
    problem.add_goal(robot_in(roomB))

    # We want to minimize the plan cost.
    problem.add_quality_metric(MinimizeActionCosts({}, default=Int(1)))
    return problem

problem = get_planning_task()


## Part (b): Find a (possibly suboptimal) plan

Solve the task with greedy best-first search using the FF heuristic. The example code below uses the h^add heuristic. You need to inspect the output to stdout to see the heuristic value of the initial state.

In [5]:
params = {
    "fast_downward_search_config": "eager_greedy([add()])"
}

with OneshotPlanner(name="fast-downward", params=params) as planner:
    result = planner.solve(problem)
    if result.status == up.engines.PlanGenerationResultStatus.SOLVED_SATISFICING:
        print("Found a plan of length:", len(result.plan.actions))
        print(result.plan)
        with PlanValidator() as validator:
            val_result = validator.validate(problem, result.plan)
            print("Plan cost:", val_result.metric_evaluations)
    else:
        print("No plan found.")

Found a plan of length: 1
SequentialPlan:
    move(roomA, doorB, roomB)
Plan cost: {minimize actions-cost: {'default': 1}: 1}


## Part (c): Find an optimal plan

Solve the task with A* using the `iPDB` heuristic.